In [1]:
import os, shlex, subprocess, json
from datetime import datetime
from pathlib import Path
from typing import Optional, Tuple, Dict
from dotenv import load_dotenv
load_dotenv(Path("configs") / "local.env")

#from data.minbpe import BasicTokenizer as Tokenizer
from src.minbpe import RegexTokenizer as Tokenizer
#from src.gpt import GPTLanguageModel
from src.transformer.model_relative_positional_encoding import GPTLanguageModel

import matplotlib.pyplot as plt
import torch
torch.manual_seed(3647)
torch.set_float32_matmul_precision('high')
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

In [2]:
def now():
    return datetime.now().astimezone().strftime('%FT%T%:z')

def print_model_structure(model: nn.Module, indent: str = '') -> None:
    """
    Custom function to print model structure in a hierarchical format
    """
    for name, child in model.named_children():
        params = sum(p.numel() for p in child.parameters())
        print(f"{indent}├─ {name}: {child.__class__.__name__} ({params:,} parameters)")
        print_model_structure(child, indent + '│  ')

def get_ckpt_files(ckpt_dir, match="checkpoint_*.pt"):
    def get_train_loss(pt_file):
        jf = str(pt_file).replace(".pt", "") + ".json"
        d = json.loads(Path(jf).read_text())
        return pt_file, d['train_loss'], d['created_at']

    result = [get_train_loss(f) for f in ckpt_dir.glob(match)]

    result = sorted(
        result,
        #key=lambda x: x.stat().st_ctime,
        key=lambda x: x[0],
        reverse=True,
    )

    return result

def send_notification(title, message):
    cmd = os.getenv("send_notification")
    if cmd is None:
        return

    command = shlex.split(cmd)
    command.append(title)
    command.append(message)

    _ = subprocess.Popen(command)


In [3]:
class TextDataset(Dataset):
    def __init__(self, data: torch.Tensor, block_size: int) -> None:
        self.data = data
        self.block_size = block_size

    def __len__(self) -> int:
        return len(self.data) - self.block_size

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        x = self.data[index:index + self.block_size]
        y = self.data[index + 1:index + self.block_size + 1]
        return x, y

def get_dataloaders(
        train_data: torch.Tensor,
        val_data: torch.Tensor,
        block_size: int,
        batch_size: int,
        device: torch.device,
) -> Tuple[DataLoader, DataLoader]:
    train_dataset = TextDataset(train_data.to(device), block_size)
    val_dataset = TextDataset(val_data.to(device), block_size)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    return train_loader, val_loader

In [4]:
#### 1. init
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer_dir = Path("data") / "tokenizer"

checkpoint_dir = Path("data") / "ch09"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

In [5]:
#### 2. setup
tokenizer = Tokenizer()
tokenizer.load(model_file=str(tokenizer_dir / "tokenizer.model"))

train_data = torch.load(tokenizer_dir / 'train.tokens.pt')
val_data = torch.load(tokenizer_dir / 'validation.tokens.pt')

print(f"--> train_data: {train_data.size()[0]:_}, val_data: {val_data.size()[0]:_}")

--> train_data: 12_092_088, val_data: 616_680


In [6]:
#### 3. parameters
##### 3.1 model parameters
parameters = {
  'vocab_size': len(tokenizer.vocab),
  'n_embd': 512,
  'block_size': 256,
  'n_head': 8,
  'n_layer': 4,
  'dropout': 0.2,
}

##### 3.2 training parameters
batch_size = 96 # 64, 32
min_learning_rate = 3e-5
max_learning_rate = 3e-4

num_epochs = 5
eval_interval = 1_000
eval_batches = 1_000 # 5_000

In [7]:
#### 4. datasets
train_loader, val_loader = get_dataloaders(
    train_data=train_data,
    val_data=val_data,
    block_size=parameters['block_size'],
    batch_size=batch_size,
    device=device,
)

eval_batches = min(eval_batches, len(val_loader))

eval_train_loader, eval_val_loader = get_dataloaders(
    train_data=train_data[:eval_batches*batch_size],
    val_data=val_data[:eval_batches*batch_size],
    block_size=parameters['block_size'],
    batch_size=batch_size,
    device=device,
)

print(f"{now()} train_batches={len(train_loader):_}, validation_batched={len(val_loader):_}, eval_batches={eval_batches:_}")

2025-09-16T05:31:17%:z train_batches=125_957, validation_batched=6_422, eval_batches=1_000


In [8]:
#### 5. Scheduler calculation
gradient_accumulation_steps = 8
epoch_last, batch_steps_last = 1, 0
batch_steps_total = len(train_loader) * num_epochs
optimizer_steps_total = batch_steps_total // gradient_accumulation_steps
warmup_iters = int(0.1 * optimizer_steps_total)

print(
    f"{now()} batch_steps_total={batch_steps_total:_}, "
    f"optimizer_steps_total={optimizer_steps_total:_}, "
    f"warmup_iters={warmup_iters:07_}"
)

2025-09-16T05:31:17%:z batch_steps_total=629_785, optimizer_steps_total=78_723, warmup_iters=007_872


In [9]:
#### 6. llm
model = GPTLanguageModel(
    vocab_size=parameters['vocab_size'],
    n_embd=parameters['n_embd'],
    block_size=parameters['block_size'],
    n_head=parameters['n_head'],
    n_layer=parameters['n_layer'],
    dropout=parameters['dropout'],
    device=device,
).to(device)

model = torch.compile(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=max_learning_rate)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer=optimizer,
    T_max=optimizer_steps_total - warmup_iters,
    eta_min=min_learning_rate,
)

parameters_m = sum(p.numel() for p in model.parameters())/1e6
print(f'Model parameters: {parameters_m:.3f}M ')

# warmup_scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=10)
# warmup_scheduler.step()

Model parameters: 13.660M 


In [10]:
#### 7. last checkpoints
ckpt_files = sorted(
    checkpoint_dir.glob("checkpoint_*.pt"),
    #key=lambda x: x.stat().st_ctime,
    key=lambda x: int(x.name.replace("checkpoint_", "").replace(".pt", "").split("-")[-1]),
    reverse=True,
)

if len(ckpt_files) > 0:
    checkpoint_path = ckpt_files[0]
    last_ckpt = torch.load(checkpoint_path, map_location=device) # weights_only=True
    epoch_last = last_ckpt['meta']['epoch']
    batch_steps_last = last_ckpt['meta']['step']

    print(f"load: last_checkpoint={checkpoint_path}, batch_steps_last={batch_steps_last:07_}")    
    model.load_state_dict(last_ckpt['model_state_dict'])

    optimizer.load_state_dict(last_ckpt['optimizer_state_dict'])

    scheduler.load_state_dict(last_ckpt['scheduler_state_dict'])
    scheduler.T_max = optimizer_steps_total - warmup_iters
    scheduler.eta_min = min_learning_rate

batch_processed = (epoch_last - 1) * len(train_loader)
optimizer_processed = batch_steps_last // gradient_accumulation_steps

print(f"batch_steps_last={batch_steps_last:_}/{batch_steps_total:_}, optimizer_processed={optimizer_processed:_}")

load: last_checkpoint=data/ch09/checkpoint_002-152000.pt, batch_steps_last=152_000
batch_steps_last=152_000/629_785, optimizer_processed=19_000


In [11]:
#### 8. estimate losses
@torch.no_grad()
def estimate_loss() -> Dict[str, float]:
    output = {}
    model.eval() # 评估模式

    for split, loader in [('train', eval_train_loader), ('val', eval_val_loader)]:
        losses = torch.zeros(eval_batches)
        for i, (x, y) in enumerate(loader):
            with torch.no_grad():
                _, loss = model(x, y)
            losses[i] = loss.item()

        output[split] = float(losses.mean().item())

    model.train() # 切换到为推理
    return output

def estimate_and_save(epoch, step):
    t0 = datetime.now()
    losses = estimate_loss()
    #current_lr = scheduler.get_last_lr()[0]
    current_lr = optimizer.param_groups[0]['lr']

    # Save checkpoint
    checkpoint_prefix = str(checkpoint_dir / f"checkpoint_{epoch:03d}-{step:06d}")

    meta = {
        'created_at': now(),
        'parameters': parameters,
        'epoch': epoch,
        'step': step,
        'learning_rate': current_lr,
        'train_loss': losses['train'],
        'val_loss': losses['val'],
        'batch_size': batch_size,
    }

    checkpoint = {
        'meta': meta,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
    }

    with open(checkpoint_prefix + ".json", 'w', encoding='utf-8') as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
        f.write("\n")

    torch.save(checkpoint, checkpoint_prefix+".pt")

    elapsed = datetime.now() - t0

    print(
        f"\n{now()} epoch={epoch}/{num_epochs}, step={step:07_}/{batch_steps_total:07_}, learning_rate={current_lr:.6f}, "
        f"\n    train_loss={losses['train']:.3f}, val_loss={losses['val']:.3f}, "
        f"checkpoint={checkpoint_prefix}.pt, elapsed={str(elapsed)}"
    )

    return losses

In [ ]:
#### 9. training
train_losses, val_losses = [], []


for epoch in range(epoch_last, num_epochs+1):
    print(f"{now()} start training: epoch={epoch}")
    send_notification("ch09_train", f"start training: epoch={epoch}")

    for _, (x_batch, y_batch) in enumerate(train_loader):
        batch_processed += 1
        if batch_processed <= batch_steps_last:
            continue

        # Training step
        optimizer.zero_grad(set_to_none=True)
        logits, loss = model(x_batch, y_batch)
        loss /= gradient_accumulation_steps
        loss.backward()
        #batch_loss = loss.item()

        if batch_processed % gradient_accumulation_steps != 0:
            continue

        optimizer_processed += 1
        if optimizer_processed <= warmup_iters:
            learning_rate = max_learning_rate * (optimizer_processed / warmup_iters)
            for param_group in optimizer.param_groups:
                param_group['lr'] = learning_rate

        optimizer.step()

        if optimizer_processed >= warmup_iters:
            scheduler.step()

        if optimizer_processed % eval_interval != 0:
            continue

        # current_lr = scheduler.get_last_lr()[0]
        current_lr = optimizer.param_groups[0]['lr']
        losses = estimate_and_save(epoch, step=batch_processed)
        train_losses.append(losses['train'])
        val_losses.append(losses['val'])

        msg = "{} evaluation: epoch={}, batch_processed={}, train_loss={:.3f}, validation_loss={:.3f}, learning_rate={:.6f}".format(
            now(), epoch, batch_processed, losses['train'], losses['val'], current_lr,
        )

        send_notification("ch09_train", msg)

    epoch_pt = checkpoint_dir / f"checkpoint_{epoch:03d}-{batch_processed:06d}.pt"
    if not epoch_pt.exists():
        current_lr = optimizer.param_groups[0]['lr']
        losses = estimate_and_save(epoch, step=batch_processed)
        train_losses.append(losses['train'])
        val_losses.append(losses['val'])

send_notification("ch09_train", "Training finished")

2025-09-16T05:31:19%:z start training: epoch=2
{"status":1,"request":"95635392-400d-4d7b-9c80-6043cdacefc4"}

In [ ]:
#### 10. data viz
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="Training Loss") # marker='o'
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Evaluation Step")
plt.ylabel("Loss")
plt.title("Training and Validation Loss Over Time")
plt.legend()
plt.grid()
plt.show()